# Ingestion Pipeline


In [17]:
import os
import sys
import json
import pandas as pd
from pathlib import Path

# Add src to path
sys.path.insert(0, 'src')

import glob
from abbreviator import Abbreviator
from create_docs import prepare_docs
from upload_dag import create_upload_dag_sh
from process_se15_file import map_se15_files_to_dataframe
from create_dags import prepare_dag_file, prepare_sql_file
from utils import check_and_remove_duplicates, add_derived_columns
from parser import parse_all_se15_files, save_to_json, DataSensitivityClassifier
from bucket import generate_bucket_input_csv, parse_final_bucket_info, merge_bucket_ids
from config import (SE15_FILES_DIR, INPUT_CSV, SE15_TABLES_JSON,OUTPUT_CONFIGS_DIR, WORD_ABBREVIATIONS_JSON,get_bucket_paths, ensure_dir_exists)

# Pandas display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 1000)

# Auto-reload modules when they change (use this if you're making changes to source files)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---
# STEP 1: Parse SE15 Files

Parse SAP SE15 table definition files to extract:
- Table names and descriptions
- Column definitions
- Data types and constraints

Then generate:
- Abbreviated table names
- Data sensitivity classifications

In [18]:
# Parse all SE15 files
results = parse_all_se15_files(str(SE15_FILES_DIR))

if results['processed_tables'] == 0:
    raise Exception(" No tables parsed. Check SE15 files directory.")

# Save initial results
save_to_json(results['tables'], str(SE15_TABLES_JSON))

Processing: BUT000.txt
  ✓ Parsed BUT000: 96 columns
Processing: SCDL DB_DATE.txt
  ✓ Parsed SCDL_DB_DATE: 11 columns
Processing: SCDL DB_PROCH_I.txt
  ✓ Parsed SCDL_DB_PROCH_I: 39 columns
Processing: SCDL DB_PROCH_O.txt
  ✓ Parsed SCDL_DB_PROCH_O: 42 columns
Processing: SCDL DB_PROCI_I.txt
  ✓ Parsed SCDL_DB_PROCI_I: 138 columns
Processing: SCDL DB_PROCI_O.txt
  ✓ Parsed SCDL_DB_PROCI_O: 142 columns
Processing: SCDL DB_REFDOC.txt
  ✓ Parsed SCDL_DB_REFDOC: 14 columns
Processing: SCDL DB_REQI.txt
  ✓ Parsed SCDL_DB_REQI: 81 columns
Processing: SCDL DB_STATUS.txt
  ✓ Parsed SCDL_DB_STATUS: 10 columns
Processing: SCWM AQUA.txt
  ✓ Parsed SCWM_AQUA: 41 columns
Processing: SCWM BINMAT.txt
  ✓ Parsed SCWM_BINMAT: 17 columns
Processing: SCWM ORDIM_C.txt
  ✓ Parsed SCWM_ORDIM_C: 156 columns
Processing: SCWM ORDIM_H.txt
  ✓ Parsed SCWM_ORDIM_H: 45 columns
Processing: SCWM ORDIM_O.txt
  ✓ Parsed SCWM_ORDIM_O: 157 columns
Processing: SCWM QUAN.txt
  ✓ Parsed SCWM_QUAN: 34 columns
Processing: SCW

True

### Generate Table Abbreviations

In [19]:
abbr = Abbreviator(str(WORD_ABBREVIATIONS_JSON))
for table in results['tables']:
    table_name = table.get('table_name', '')
    table_desc = table.get('table_description', '')
    table['table_abbrev'] = abbr.generate_abbreviation(
        table_desc, 
        context=table_name, 
        is_table=True
    )
    for column in table.get('columns', []):
        col_desc = column.get('description', '')
        column['column_abbrev'] = abbr.generate_abbreviation(
            col_desc,
            context=None,
            is_table=False
        )
        
    table['columns'].append({
    "column_name": "DS_LOAD_START_TS",
    "column_abbrev": "ds_load_ts",
    "sensitivity_level": "",
    "description": "DS Load Start Timestamp",
    "is_primary_key": False,
    "data_type": "TIMESTAMP",   # you can change this if needed
    "length": "0"
})    
print(f"   Generated abbreviations for {results['processed_tables']} tables")

# Show sample abbreviations
print("\n Sample Table Abbreviations:")
for table in results['tables'][:3]:
    print(f"  {table['table_name']:20} → {table['table_abbrev']}")

   Generated abbreviations for 21 tables

 Sample Table Abbreviations:
  BUT000               → bp_gen_data_i
  SCDL_DB_DATE         → dt
  SCDL_DB_PROCH_I      → inbound_dlvr_hdr


### Classify Data Sensitivity

In [20]:
classifier = DataSensitivityClassifier()
sensitivity_stats = {'hs': 0, 'ns': 0, 'se': 0}

for table in results['tables']:
    sensitivity = classifier.classify_table(table)
    table['sensitivity_level'] = sensitivity
    sensitivity_stats[sensitivity] += 1

print(f"  Classified {results['processed_tables']} tables:")
print(f"    - Highly Sensitive (hs): {sensitivity_stats['hs']}")
print(f"    - Non-Sensitive (ns):    {sensitivity_stats['ns']}")
print(f"    - Needs Evaluation (se): {sensitivity_stats['se']}")

# Save final version with abbreviations and sensitivity
save_to_json(results['tables'], str(SE15_TABLES_JSON))

  Classified 21 tables:
    - Highly Sensitive (hs): 1
    - Non-Sensitive (ns):    3
    - Needs Evaluation (se): 17
  ✓ Saved: /Users/vn59efm/Desktop/latest_automation/Automation/output/se15_tables.json


True

---
# STEP 2: Load & Merge Configuration

Merge SE15 metadata with ingestion configuration.

In [21]:
# Load SE15 metadata
print(f"\n Loading SE15 metadata from {SE15_TABLES_JSON}")
with open(SE15_TABLES_JSON, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Create dataframe with SE15 data including columns
tables_data = []
for table in data:
    # Extract column mapping (column_name -> column_abbrev)
    column_mapping = {}
    for column in table.get('columns', []):
        col_name = column.get('column_name', '')
        col_abbrev = column.get('column_abbrev', '')
        if col_name and col_abbrev:
            column_mapping[col_name] = col_abbrev
    
    tables_data.append({
        'icdsTableName': table['table_name'],
        'dlTableName': table['table_abbrev'],
        'table_description': table['table_description'],
        'dataSensitivity': table['sensitivity_level'],
        'column_mapping': column_mapping,  # Add column mapping
        'columns': table.get('columns', [])  # Add full column data for SQL generation
    })

df_with_table_name = pd.DataFrame(tables_data)
print(f"   Loaded {len(df_with_table_name)} tables from SE15")

# Load ingestion CSV
print(f"\n Loading ingestion configuration from {INPUT_CSV}")
df_without_table_name = pd.read_csv(INPUT_CSV)
print(f"   Loaded {len(df_without_table_name)} records from CSV")

# Preview ingestion config
print("\n Ingestion Config Sample:")
display(df_without_table_name.head(3))



 Loading SE15 metadata from /Users/vn59efm/Desktop/latest_automation/Automation/output/se15_tables.json
   Loaded 21 tables from SE15

 Loading ingestion configuration from /Users/vn59efm/Desktop/latest_automation/Automation/input/ingestion.csv
   Loaded 47 records from CSV

 Ingestion Config Sample:


,icdsTableName,BANNER_NAME,OP-Company code,dlSchemaName,tableLoadType
0,SCWM_TU_STATUS,MM,SA-MM,sa_supply_chain_dl_secure,INC
1,SCWM_TU_DLV,MM,SA-MM,sa_supply_chain_dl_secure,INC
2,SCDL_DB_PROCH_I,MM,SA-MM,sa_supply_chain_dl_secure,INC


### Check for Duplicates

In [22]:
df_without_table_name, duplicate_records, removed_count, dup_summary = check_and_remove_duplicates(
    df_without_table_name, 
    column='icdsTableName', 
    keep='first'
)


Total duplicate records: 0
Unique duplicate values: 0

✓ No duplicates found in 'icdsTableName'!


### Merge Dataframes

In [23]:
print("\n Merging SE15 metadata with ingestion config...")
df = df_with_table_name.merge(
    df_without_table_name, 
    on='icdsTableName', 
    how='left'
)
print(f"   Merged dataframe has {len(df)} rows")

# Map SE15 files
print("\n Mapping SE15 files...")
df = map_se15_files_to_dataframe(df)

se15_count = df['se15_file_exists'].sum()
no_se15_count = len(df) - se15_count
print(f"   Records with SE15 files: {se15_count}")
print(f"    Records without SE15 files: {no_se15_count}")

# Filter to only rows with SE15 files
df = df.loc[df['se15_file_exists']].reset_index(drop=True)
print(f"\n Processing {len(df)} tables with SE15 files")

# Add derived columns
print("\n  Adding derived columns...")
df = add_derived_columns(df)
print("   Added: cluster_name, table_name, dag_name, output_dir, tags")

print("\n Final Dataframe:")
display(df.head(3))


 Merging SE15 metadata with ingestion config...
   Merged dataframe has 21 rows

 Mapping SE15 files...
   Records with SE15 files: 21
    Records without SE15 files: 0

 Processing 21 tables with SE15 files

  Adding derived columns...
   Added: cluster_name, table_name, dag_name, output_dir, tags

 Final Dataframe:


,icdsTableName,dlTableName,table_description,dataSensitivity,column_mapping,columns,BANNER_NAME,OP-Company code,dlSchemaName,tableLoadType,se15_file_exists,cluster_name,table_name,dag_name,output_dir,full_banner_name,tags
0,BUT000,bp_gen_data_i,BP: General data I,hs,"{'CLIENT': 'clnt', 'PARTNER': 'bus_prtnr_nbr',...","[{'column_name': 'CLIENT', 'column_abbrev': 'c...",MM,SA-MM,sa_supply_chain_dl_secure,INC,True,sa-supply-chain-dl-secure-mm-bp-gen-data-i,mm_bp_gen_data_i,INTLDLDAT-SAMM-INC-SA_SUPPLY_CHAIN_DL_SECURE-M...,/Users/vn59efm/Desktop/latest_automation/Autom...,massmart,"[Massmart-eComm, P2, Ephemeral, SA, MASSMART, ..."
1,SCDL_DB_DATE,dt,Date,se,"{'MANDT': 'clnt', 'DOCID': 'doc_id', 'ITEMID':...","[{'column_name': 'MANDT', 'column_abbrev': 'cl...",MM,SA-MM,sa_supply_chain_dl_secure,INC,True,sa-supply-chain-dl-secure-mm-dt,mm_dt,INTLDLDAT-SAMM-INC-SA_SUPPLY_CHAIN_DL_SECURE-M...,/Users/vn59efm/Desktop/latest_automation/Autom...,massmart,"[Massmart-eComm, P2, Ephemeral, SA, MASSMART, ..."
2,SCDL_DB_PROCH_I,inbound_dlvr_hdr,Inbound Delivery: Header,se,"{'MANDT': 'clnt', 'DOCID': 'doc_id', 'DOCCAT':...","[{'column_name': 'MANDT', 'column_abbrev': 'cl...",MM,SA-MM,sa_supply_chain_dl_secure,INC,True,sa-supply-chain-dl-secure-mm-inbound-dlvr-hdr,mm_inbound_dlvr_hdr,INTLDLDAT-SAMM-INC-SA_SUPPLY_CHAIN_DL_SECURE-M...,/Users/vn59efm/Desktop/latest_automation/Autom...,massmart,"[Massmart-eComm, P2, Ephemeral, SA, MASSMART, ..."


---
# STEP 3: Generate DAGs & Documentation

Generate DAG files and documentation (these don't require bucket IDs).
SQL files will be generated after bucket creation.

In [24]:
# Generate DAG files (independent of bucket_id)
dag_count = 0
dag_errors = []

for idx, row in df.iterrows():
    try:
        prepare_dag_file(row)
        dag_count += 1
    except Exception as e:
        error_msg = f"{row.icdsTableName}: {str(e)}"
        dag_errors.append(error_msg)
        print(f"    Error: {error_msg}")

print(f"   Generated {dag_count} DAG files")
if dag_errors:
    print(f"    {len(dag_errors)} errors occurred")

# Generate documentation (independent of bucket_id)
docs_count = 0
docs_errors = []

for idx, row in df.iterrows():
    try:
        prepare_docs(row)
        docs_count += 1
    except Exception as e:
        error_msg = f"{row.icdsTableName}: {str(e)}"
        docs_errors.append(error_msg)
        print(f"    Error: {error_msg}")

print(f"   Generated documentation for {docs_count} tables")
if docs_errors:
    print(f"    {len(docs_errors)} errors occurred")


   Generated 21 DAG files
   Generated documentation for 21 tables


---
# STEP 4: Generate Bucket Input CSV

Create environment-specific `bucket_input.csv` file for GCS bucket creation portal.

**File naming:**
- Dev: `input/buckets/dev-bucket_input.csv`
- Prod: `input/buckets/prod-bucket_input.csv`

Each environment maintains its own bucket files with env prefix.

In [25]:
# Ask user for environment
print("\n Select Environment:")
print("  1. dev (Development) [DEFAULT]")
print("  2. prod (Production)")
env_choice = input("\nEnter choice (1 or 2, press Enter for dev): ").strip()

if env_choice == '2':
    env = 'prod'
    print(" Selected: Production (prod)")
else:
    env = 'dev'
    if env_choice == '1':
        print(" Selected: Development (dev)")
    else:
        print(" Defaulting to: Development (dev)")

# Generate bucket input CSV with selected environment
bucket_csv = generate_bucket_input_csv(df, env=env)

if bucket_csv:
    # Preview the bucket input file
    bucket_df = pd.read_csv(bucket_csv)
    print("\n Bucket Input Preview:")
    display(bucket_df.head(3))
    print(f"\nTotal tables: {len(bucket_df)}")
    
    print("\n" + "="*80)
    print(" BUCKET INPUT FILE GENERATED")
    print("="*80)
    print(f"\n File saved at: {bucket_csv}")
    print("\n NEXT: Complete the MANUAL STEP below before proceeding to STEP 5")
    print("="*80)
else:
    print("\n Failed to generate bucket_input.csv")
    raise Exception("Could not generate bucket input file")



 Select Environment:
  1. dev (Development) [DEFAULT]
  2. prod (Production)



Enter choice (1 or 2, press Enter for dev):  


 Defaulting to: Development (dev)

 Generated bucket_input.csv with 21 tables
   Location: /Users/vn59efm/Desktop/latest_automation/Automation/input/buckets/dev-bucket_input.csv
   Environment: dev
   Sensitivity breakdown:
     - HS: 1 table(s)
     - NS: 3 table(s)
     - SE: 17 table(s)

 Next Step (Manual): Use this file to create buckets in the portal
   Save the portal result as: /Users/vn59efm/Desktop/latest_automation/Automation/input/buckets/FinalBucketInfo.csv

 Bucket Input Preview:


,bucketNameType,databaseName,tableName,opCmpnyCd,refreshMode,wmt.storage_uploader,wmt.storage_viewer,isDevBigLake,updateSoftDelete,resourceBucketType
0,hash,sa_supply_chain_dl_secure,mm_bp_gen_data_i,SA-MM,incremental load,svc-dl-sa-afaas-hs@wmt-intl-dl-sa-hs-dev.iam.g...,svc-dl-sa-afaas-hs@wmt-intl-dl-sa-hs-dev.iam.g...,False,DCA Logic,NaN
1,hash,sa_supply_chain_dl_secure,mm_dt,SA-MM,incremental load,svc-dl-sa-afaas-se@wmt-intl-dl-sa-se-dev.iam.g...,svc-dl-sa-afaas-se@wmt-intl-dl-sa-se-dev.iam.g...,False,DCA Logic,NaN
2,hash,sa_supply_chain_dl_secure,mm_inbound_dlvr_hdr,SA-MM,incremental load,svc-dl-sa-afaas-se@wmt-intl-dl-sa-se-dev.iam.g...,svc-dl-sa-afaas-se@wmt-intl-dl-sa-se-dev.iam.g...,False,DCA Logic,NaN



Total tables: 21

 BUCKET INPUT FILE GENERATED

 File saved at: /Users/vn59efm/Desktop/latest_automation/Automation/input/buckets/dev-bucket_input.csv

 NEXT: Complete the MANUAL STEP below before proceeding to STEP 5


In [26]:



print("="*80)
print("STEP 5: Processing Bucket IDs")
print("="*80)

# Get environment-specific path (should match the env used in STEP 4)
bucket_paths = get_bucket_paths(env)
FINAL_BUCKET_INFO_CSV = bucket_paths['final_bucket_info_csv']

# Check if FinalBucketInfo.csv exists
if not FINAL_BUCKET_INFO_CSV.exists():
    print(f"\n ERROR: FinalBucketInfo.csv not found!")
    print(f"   Expected location: {FINAL_BUCKET_INFO_CSV}")
    print("\n" + "="*80)
    print(" CANNOT PROCEED - MANUAL STEP INCOMPLETE")
    print("="*80)
    print("\nYou must complete the manual bucket creation step:")
    print("1. Upload bucket_input.csv to the portal")
    print("2. Download the FinalBucketInfo.csv response")
    print(f"3. Save it at: {FINAL_BUCKET_INFO_CSV}")
    print("\nThen re-run this cell to continue.")
    print("="*80)
    raise FileNotFoundError(f"FinalBucketInfo.csv not found at {FINAL_BUCKET_INFO_CSV}")

print(f"\n Found FinalBucketInfo.csv")

try:
    # Parse FinalBucketInfo.csv with environment
    bucket_mapping = parse_final_bucket_info(env=env)
    
    # Preview bucket mapping
    print("\n Bucket Mapping Preview:")
    display(bucket_mapping.head(10))
    
    # Drop existing bucket_id column if present (for re-runs)
    if 'bucket_id' in df.columns:
        df = df.drop(columns=['bucket_id'])
    
    # Merge bucket IDs into main dataframe
    df = merge_bucket_ids(df, bucket_mapping)
    
    # Show sample with bucket IDs
    print("\n Dataframe with Bucket IDs:")
    display(df[['icdsTableName', 'table_name', 'bucket_id']].head(10))
    
    # Regenerate JSON metadata files with bucket IDs
    print("\n Updating JSON metadata files with bucket IDs...")
    json_count = 0
    for idx, row in df.iterrows():
        try:
            from create_docs import create_column_mapping_file
            output_dir = row.output_dir
            icds = row.get('icdsTableName') if hasattr(row, 'get') else getattr(row, 'icdsTableName', None)
            docs_dir_name = f"docs-{icds}" if icds else 'docs'
            docs_dir = os.path.join(output_dir, docs_dir_name)
            create_column_mapping_file(row, docs_dir)
            json_count += 1
        except Exception as e:
            print(f"    Error updating {row.icdsTableName}: {str(e)}")
    
    print(f"   Updated {json_count} JSON metadata files with bucket IDs")
    
    print("\n Bucket IDs successfully merged!")
    
except Exception as e:
    print(f"\n Error processing FinalBucketInfo.csv: {e}")
    print("\nPlease check:")
    print("1. File format is correct (CSV with tableName, bucket_name columns)")
    print("2. File is not corrupted")
    print("3. Table names match the ones in bucket_input.csv")
    raise


STEP 5: Processing Bucket IDs

 Found FinalBucketInfo.csv

 Parsed FinalBucketInfo.csv
   Found 21 bucket mappings

 Bucket Mapping Preview:


,tableName,bucket_id
0,bp_data,c4efafb620253e78ccdf14b7d3f4067df049d4e16c5242...
1,dt_scdl_db,325f2d33d27159319c9f7bb484540e7381e4f313ca5d55...
2,dlvr_hdr,d5b0e0f086b27fa71182a7c89debd03fbce6966eaa748f...
3,dlvr_order_hdr,fbe5ce334b283c6a14ec13a7e9e5165dbe8bce82affd07...
4,inbound_dlvr_item,ac410d88a43cdacdbe66773967f6a0b65debc5870add89...
5,dlvr_order_item,8b65b6a7dcb9aadde3908b7cc5159e10bee5e77c554582...
6,ref_scdl_db,89f6b8e1df674b4a10d97b8e800156d9f79aa94d8c7b5e...
7,item_notif_rq,3983f199dcdeeb2961485d344cf881f59fc41efbe0e016...
8,status_scdl_db,4cb116244d7afb73ef91c92a35177137578e4964140534...
9,avlbl_qty,33709e459c967c07980b8c9326ced9e4888f23a5eaeee2...



🔗 Merged bucket IDs into dataframe:
   ✓ Tables with bucket_id: 3
    Tables without bucket_id: 18
   Missing tables: mm_bp_gen_data_i, mm_dt, mm_inbound_dlvr_hdr, mm_outbound_dlvr_order_hdr, mm_outbound_dlvr_order_item

 Dataframe with Bucket IDs:


,icdsTableName,table_name,bucket_id
0,BUT000,mm_bp_gen_data_i,NaN
1,SCDL_DB_DATE,mm_dt,NaN
2,SCDL_DB_PROCH_I,mm_inbound_dlvr_hdr,NaN
3,SCDL_DB_PROCH_O,mm_outbound_dlvr_order_hdr,NaN
4,SCDL_DB_PROCI_I,mm_inbound_dlvr_item,ac410d88a43cdacdbe66773967f6a0b65debc5870add89...
5,SCDL_DB_PROCI_O,mm_outbound_dlvr_order_item,NaN
6,SCDL_DB_REFDOC,mm_ref,NaN
7,SCDL_DB_REQI,mm_item_inbound_dlvr_notif_outbound_del_rq,NaN
8,SCDL_DB_STATUS,mm_status,NaN
9,SCWM_AQUA,mm_avlbl_qty,33709e459c967c07980b8c9326ced9e4888f23a5eaeee2...



 Updating JSON metadata files with bucket IDs...
   Updated 21 JSON metadata files with bucket IDs

 Bucket IDs successfully merged!


---
# STEP 6: Generate SQL Files

Generate SQL files with bucket IDs from FinalBucketInfo.csv.

**Prerequisites:**
- Bucket IDs loaded from STEP 5

In [25]:
# Check if bucket_id column exists
if 'bucket_id' not in df.columns or df['bucket_id'].isna().all():
    print("\n ERROR: No bucket IDs found in dataframe!")
    print("   Please complete STEP 5 first to load bucket IDs.")
    raise ValueError("bucket_id column is missing or empty. Run STEP 5 first.")

# Generate SQL files with bucket IDs
sql_count = 0
sql_errors = []

for idx, row in df.iterrows():
    try:
        prepare_sql_file(row)
        sql_count += 1
    except Exception as e:
        error_msg = f"{row.icdsTableName}: {str(e)}"
        sql_errors.append(error_msg)
        print(f"    Error: {error_msg}")

print(f"   Generated {sql_count} SQL files with bucket IDs")
if sql_errors:
    print(f"   {len(sql_errors)} errors occurred")



   Generated 21 SQL files with bucket IDs


## create upload commands


In [26]:
DEST_DAG_BUCKET = "gs://bfdaf-dags-intldlsadev-catalog/"
dag_files = sorted(glob.glob(f"{OUTPUT_CONFIGS_DIR}/**/*.py", recursive=True))
output_path = "output/upload_dag.sh"

create_upload_dag_sh(dag_files, DEST_DAG_BUCKET, output_path)
print(f" upload commands created at : ",output_path)

 upload commands created at :  output/upload_dag.sh


---
# Final Dataframe Info

In [27]:
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSample Records:")
display(df.head(10))

Shape: (21, 18)

Columns: ['icdsTableName', 'dlTableName', 'table_description', 'dataSensitivity', 'column_mapping', 'columns', 'BANNER_NAME', 'OP-Company code', 'dlSchemaName', 'tableLoadType', 'se15_file_exists', 'cluster_name', 'table_name', 'dag_name', 'output_dir', 'full_banner_name', 'tags', 'bucket_id']

Sample Records:


,icdsTableName,dlTableName,table_description,dataSensitivity,column_mapping,columns,BANNER_NAME,OP-Company code,dlSchemaName,tableLoadType,se15_file_exists,cluster_name,table_name,dag_name,output_dir,full_banner_name,tags,bucket_id
0,BUT000,bp_gen_data,BP: General data I,hs,"{'CLIENT': 'clnt', 'PARTNER': 'bus_prtnr_nbr',...","[{'column_name': 'CLIENT', 'column_abbrev': 'c...",MM,SA-MM,sa_supply_chain_dl_secure,INC,True,sa-supply-chain-dl-secure-mm-bp-gen-data,mm_bp_gen_data,INTLDLDAT-SAMM-INC-SA_SUPPLY_CHAIN_DL_SECURE-M...,/Users/vn59efm/Desktop/latest_automation/Autom...,massmart,"[Massmart-eComm, P2, Ephemeral, SA, SECURE, MD...",NaN
1,SCDL_DB_DATE,dt_scdl_db,Date,se,"{'MANDT': 'clnt', 'DOCID': 'doc_id', 'ITEMID':...","[{'column_name': 'MANDT', 'column_abbrev': 'cl...",MM,SA-MM,sa_supply_chain_dl_secure,INC,True,sa-supply-chain-dl-secure-mm-dt-scdl-db,mm_dt_scdl_db,INTLDLDAT-SAMM-INC-SA_SUPPLY_CHAIN_DL_SECURE-M...,/Users/vn59efm/Desktop/latest_automation/Autom...,massmart,"[Massmart-eComm, P2, Ephemeral, SA, SECURE, MD...",325f2d33d27159319c9f7bb484540e7381e4f313ca5d55...
2,SCDL_DB_PROCH_I,inbound_dlvr_hdr,Inbound Delivery: Header,se,"{'MANDT': 'clnt', 'DOCID': 'doc_id', 'DOCCAT':...","[{'column_name': 'MANDT', 'column_abbrev': 'cl...",MM,SA-MM,sa_supply_chain_dl_secure,INC,True,sa-supply-chain-dl-secure-mm-inbound-dlvr-hdr,mm_inbound_dlvr_hdr,INTLDLDAT-SAMM-INC-SA_SUPPLY_CHAIN_DL_SECURE-M...,/Users/vn59efm/Desktop/latest_automation/Autom...,massmart,"[Massmart-eComm, P2, Ephemeral, SA, SECURE, MD...",NaN
3,SCDL_DB_PROCH_O,outbound_order_hdr,Outbound Delivery Order Header,se,"{'MANDT': 'clnt', 'DOCID': 'doc_id', 'DOCCAT':...","[{'column_name': 'MANDT', 'column_abbrev': 'cl...",MM,SA-MM,sa_supply_chain_dl_secure,INC,True,sa-supply-chain-dl-secure-mm-outbound-order-hdr,mm_outbound_order_hdr,INTLDLDAT-SAMM-INC-SA_SUPPLY_CHAIN_DL_SECURE-M...,/Users/vn59efm/Desktop/latest_automation/Autom...,massmart,"[Massmart-eComm, P2, Ephemeral, SA, SECURE, MD...",NaN
4,SCDL_DB_PROCI_I,inbound_dlvr_item,Inbound Delivery Item,se,"{'MANDT': 'clnt', 'DOCID': 'doc_id', 'ITEMID':...","[{'column_name': 'MANDT', 'column_abbrev': 'cl...",MM,SA-MM,sa_supply_chain_dl_secure,INC,True,sa-supply-chain-dl-secure-mm-inbound-dlvr-item,mm_inbound_dlvr_item,INTLDLDAT-SAMM-INC-SA_SUPPLY_CHAIN_DL_SECURE-M...,/Users/vn59efm/Desktop/latest_automation/Autom...,massmart,"[Massmart-eComm, P2, Ephemeral, SA, SECURE, MD...",ac410d88a43cdacdbe66773967f6a0b65debc5870add89...
5,SCDL_DB_PROCI_O,outbound_order_item,Outbound Delivery Order Item,se,"{'MANDT': 'clnt', 'DOCID': 'doc_id', 'ITEMID':...","[{'column_name': 'MANDT', 'column_abbrev': 'cl...",MM,SA-MM,sa_supply_chain_dl_secure,INC,True,sa-supply-chain-dl-secure-mm-outbound-order-item,mm_outbound_order_item,INTLDLDAT-SAMM-INC-SA_SUPPLY_CHAIN_DL_SECURE-M...,/Users/vn59efm/Desktop/latest_automation/Autom...,massmart,"[Massmart-eComm, P2, Ephemeral, SA, SECURE, MD...",NaN
6,SCDL_DB_REFDOC,ref_scdl_db,Reference,se,"{'MANDT': 'clnt', 'DOCID': 'doc_id', 'ITEMID':...","[{'column_name': 'MANDT', 'column_abbrev': 'cl...",MM,SA-MM,sa_supply_chain_dl_secure,INC,True,sa-supply-chain-dl-secure-mm-ref-scdl-db,mm_ref_scdl_db,INTLDLDAT-SAMM-INC-SA_SUPPLY_CHAIN_DL_SECURE-M...,/Users/vn59efm/Desktop/latest_automation/Autom...,massmart,"[Massmart-eComm, P2, Ephemeral, SA, SECURE, MD...",89f6b8e1df674b4a10d97b8e800156d9f79aa94d8c7b5e...
7,SCDL_DB_REQI,item_notif_rq,Item Inbound Delivery Notification / Outbound ...,se,"{'MANDT': 'clnt', 'DOCID': 'doc_id', 'ITEMID':...","[{'column_name': 'MANDT', 'column_abbrev': 'cl...",MM,SA-MM,sa_supply_chain_dl_secure,INC,True,sa-supply-chain-dl-secure-mm-item-notif-rq,mm_item_notif_rq,INTLDLDAT-SAMM-INC-SA_SUPPLY_CHAIN_DL_SECURE-M...,/Users/vn59efm/Desktop/latest_automation/Autom...,massmart,"[Massmart-eComm, P2, Ephemeral, SA, SECURE, MD...",3983f199dcdeeb2961485d344cf881f59fc41efbe0e016...
8,SCDL_DB_STATUS,status_scdl_db,Status,se,"{'MANDT': 'clnt', 'DOCID': 'doc_id', 'ITEMID':...","[{'column_name': 

In [28]:
# Save the dataframe in JSON format
df.to_csv("output/final_df.csv", index=False)